# Private real-MRI inventory and gated training
Default action is read-only inventory. Never infer labels or approvals from filenames.
Use the same reviewed code-only bundle as notebooks 05/06/07. Save a private notebook
copy, disable output saving, select Python 3.12 and T4 GPU for explicit training.
Set BUNDLE_PATH, independently retained BUNDLE_SHA256, and CONFIG_PATH in the mount
cell. CONFIG_PATH is a private copy of the task-specific readiness template, with
actual review evidence. No data is downloaded or uploaded by this notebook.
After Restart session rerun from top; no manual CODE_ROOT restoration is needed.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive').resolve()
BUNDLE_PATH = DRIVE_BASE / 'reviewed_code/task-ab-code.zip'  # Set your code-only ZIP.
BUNDLE_SHA256 = ''  # Required SHA-256 recorded independently when building the ZIP.
CONFIG_PATH = None  # Later: Path to your PRIVATE JSON config; never print its contents.


In [ ]:
import hashlib
import json
import stat
import zipfile

if (len(BUNDLE_SHA256) != 64 or any(c not in '0123456789abcdef' for c in BUNDLE_SHA256)
        or not BUNDLE_PATH.resolve().is_relative_to(DRIVE_BASE)
        or not BUNDLE_PATH.is_file() or BUNDLE_PATH.stat().st_size > 20 * 1024 * 1024):
    raise RuntimeError('Configure the private reviewed code bundle and independent SHA-256.')
blob = BUNDLE_PATH.read_bytes()
if hashlib.sha256(blob).hexdigest() != BUNDLE_SHA256:
    raise RuntimeError('Code bundle checksum mismatch.')
CODE_ROOT = Path('/content') / ('task_ab_code_' + BUNDLE_SHA256[:16])
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    entries = archive.infolist()
    names = [entry.filename for entry in entries]
    if (len(entries) > 64 or len(set(names)) != len(names)
            or sum(entry.file_size for entry in entries) > 20 * 1024 * 1024):
        raise RuntimeError('Code bundle resource or duplicate-entry check failed.')
    for entry in entries:
        name = Path(entry.filename)
        if (name.is_absolute() or '..' in name.parts or '\\' in entry.filename
                or stat.S_ISLNK(entry.external_attr >> 16) or entry.file_size > 2 * 1024 * 1024):
            raise RuntimeError('Unsafe code bundle member.')
    manifest = json.loads(archive.read('code_manifest.json'))
    if (manifest.get('code_only') is not True or manifest.get('schema_version') != 1
            or set(names) != set(manifest['files']) | {'code_manifest.json'}):
        raise RuntimeError('Code-only allowlist check failed.')
    # Persist the verified manifest too; restart cells need it on disk.
    verified_files = {**manifest['files'], 'code_manifest.json':
                      hashlib.sha256(archive.read('code_manifest.json')).hexdigest()}
    for name, expected in verified_files.items():
        content = archive.read(name)
        if hashlib.sha256(content).hexdigest() != expected:
            raise RuntimeError('Bundled source checksum mismatch.')
        target = CODE_ROOT / name
        if any(parent.is_symlink() for parent in (target, *target.parents)):
            raise RuntimeError('Code destination symlink rejected.')
        if target.exists():
            if target.read_bytes() != content:
                raise RuntimeError('Existing code snapshot differs; use a fresh runtime.')
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
print({'reviewed_code_verified': True})

STATE_PATH = Path('/content/brainagenticthon-code-state.json')
STATE_PATH.write_text(json.dumps({'code_root': str(CODE_ROOT), 'bundle_sha256': BUNDLE_SHA256,
    'manifest_sha256': hashlib.sha256(
        (CODE_ROOT / 'code_manifest.json').read_bytes()).hexdigest()}))


In [ ]:
# ruff: noqa: E402
import hashlib
import json
import sys
from pathlib import Path

# Survives Restart session: recover and revalidate the reviewed snapshot.
state = json.loads(Path('/content/brainagenticthon-code-state.json').read_text())
CODE_ROOT = Path(state['code_root'])
manifest_blob = (CODE_ROOT / 'code_manifest.json').read_bytes()
if hashlib.sha256(manifest_blob).hexdigest() != state['manifest_sha256']:
    raise RuntimeError('Reviewed manifest changed; rerun bundle verification.')
manifest = json.loads(manifest_blob)
for name, expected in manifest['files'].items():
    target = CODE_ROOT / name
    if (Path(name).is_absolute() or '..' in Path(name).parts
            or target.is_symlink() or not target.resolve().is_relative_to(CODE_ROOT.resolve())
            or hashlib.sha256(target.read_bytes()).hexdigest() != expected):
        raise RuntimeError('Reviewed snapshot changed; rerun bundle verification.')
import subprocess

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('This pinned research environment requires Python 3.12.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r',
                str(CODE_ROOT / 'requirements-task-a.txt')], check=True)
sys.path.insert(0, str(CODE_ROOT / 'src'))

# Check this research profile; Colab may contain unrelated incompatible packages.
from importlib.metadata import version

for requirement in (CODE_ROOT / 'requirements-task-a.txt').read_text().splitlines():
    if requirement.strip() and not requirement.startswith('#'):
        package, expected = requirement.strip().split('==')
        if version(package).split('+')[0] != expected:
            raise RuntimeError('Installed research dependency differs from reviewed pin.')


In [ ]:
# ruff: noqa: E402
import hashlib
import json
import sys
from pathlib import Path

# Survives Restart session: recover and revalidate the reviewed snapshot.
state = json.loads(Path('/content/brainagenticthon-code-state.json').read_text())
CODE_ROOT = Path(state['code_root'])
manifest_blob = (CODE_ROOT / 'code_manifest.json').read_bytes()
if hashlib.sha256(manifest_blob).hexdigest() != state['manifest_sha256']:
    raise RuntimeError('Reviewed manifest changed; rerun bundle verification.')
manifest = json.loads(manifest_blob)
for name, expected in manifest['files'].items():
    target = CODE_ROOT / name
    if (Path(name).is_absolute() or '..' in Path(name).parts
            or target.is_symlink() or not target.resolve().is_relative_to(CODE_ROOT.resolve())
            or hashlib.sha256(target.read_bytes()).hexdigest() != expected):
        raise RuntimeError('Reviewed snapshot changed; rerun bundle verification.')
sys.path.insert(0, str(CODE_ROOT / 'src'))
import nibabel as nib
import numpy as np
import torch

import brainagenticthon

if not Path(brainagenticthon.__file__).resolve().is_relative_to(CODE_ROOT):
    raise RuntimeError('Previously imported project differs from reviewed bundle; restart runtime.')
if (torch.__version__.split('+')[0] != '2.6.0'
        or np.__version__ != '2.5.4' or nib.__version__ != '5.4.2'):
    raise RuntimeError('Runtime imports differ from pinned packages; restart the runtime.')
print({'dependencies_verified': True, 'cuda_available': torch.cuda.is_available()})


In [ ]:
import uuid
from datetime import datetime, timezone

TASK = 'a'  # 'a' or 'b'; use a separate private configuration and run directory per task.
ACTION = 'inventory'  # inventory, validate, freeze, smoke, train, resume, evaluate
if CONFIG_PATH is None or not Path(CONFIG_PATH).resolve().is_relative_to(DRIVE_BASE):
    raise RuntimeError('Set CONFIG_PATH to your task readiness configuration on private Drive.')
from brainagenticthon.execution.inventory import load_config

CONFIG, PRIVATE_ROOT = load_config(CONFIG_PATH)
if PRIVATE_ROOT == DRIVE_BASE or not PRIVATE_ROOT.is_relative_to(DRIVE_BASE):
    raise RuntimeError('Use a dedicated authorized private Drive project root.')
RUN_NAME = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]


In [ ]:
from brainagenticthon.execution.inventory import save_private_report
from brainagenticthon.execution.real_training import execute_real, freeze_splits
from brainagenticthon.preprocessing.oasis2_mri_quality import private_path

if ACTION in {'inventory', 'validate'}:
    REPORT_PATH = PRIVATE_ROOT / ('task_' + TASK) / 'readiness_reports' / (RUN_NAME + '.json')
    REPORT = save_private_report(CONFIG_PATH, REPORT_PATH, task=TASK, scan=ACTION == 'inventory')
    print({'status': REPORT['status'], 'task': TASK, 'blockers': REPORT['blockers'],
           'summary': REPORT.get('summary'), 'clinical_approval': False})
elif ACTION == 'freeze':
    if 'REPLACE_WITH' in CONFIG['freeze']:
        raise RuntimeError('Choose a new immutable private freeze directory.')
    print(freeze_splits(CONFIG_PATH, private_path(PRIVATE_ROOT, CONFIG['freeze']), task=TASK))
elif ACTION in {'smoke', 'train', 'resume', 'evaluate'}:
    RESULT = execute_real(CONFIG_PATH, private_path(PRIVATE_ROOT, CONFIG['freeze']),
                          task=TASK, action=ACTION)
    print({'status': RESULT['status'], 'task': TASK, 'action': ACTION,
           'clinical_approval': False})
else:
    raise RuntimeError('Choose an explicit supported action.')


Inspect private report records for individual blockers and supporting references. Scan
headers/hashes establish technical facts only, never modality identity, true normality,
whole-brain coverage, diagnosis, licensing or clinical approval. Scan files never become
automatic manifest rows. Supply reviewed manifests/evidence and existing patient splits.
Freeze only succeeds after the existing full gate passes; it copies metadata and splits
privately, never assigns patients or changes labels. Any changed data/review/profile or
authorization invalidates the freeze.
For an approved GPU smoke use a fresh output_run, explicitly enable real_training_enabled,
and choose smoke. It reads one training batch/patch only and does not report held-out
performance. Use a different fresh output_run for train. For resume keep the same run,
set resume/trusted_resume=true and increase total epochs. For final evaluation set
final_test_evaluation=true and trusted_resume=true; evaluate does not train and final
evaluation prevents continuing that run. All gates and frozen fingerprints are rechecked
before every action. Existing checkpoint provenance/trust requirements remain.
No approved inputs are bundled. See docs/task_ab_real_data_workflow.md for exact schemas,
evidence requirements, output boundaries and remaining clinical/source-policy blockers.
